# Libraries

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from arch.unitroot import ADF, DFGLS, PhillipsPerron, ZivotAndrews

from itertools import combinations

from statsmodels.tsa.stattools import kpss, coint, grangercausalitytests

from statsmodels.tsa.api import VAR
from statsmodels.tsa.vector_ar.vecm import VECM, select_order, select_coint_rank

In [ ]:
# from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.seasonal import STL

In [ ]:
# from scipy.stats import kendalltau, pearsonr, spearmanr
# from scipy.stats import shapiro, normaltest, anderson, ks_1samp, norm
# from scipy.stats import skew, kurtosis

# Constant USD

In [ ]:
# loading up the data
data_df = pd.read_excel(r'model_data\quarterly_const_USD.xlsx', index_col='Report Date')

In [ ]:
data_df['Country Name'] = data_df['Country Name'].str.replace('China','Китай').str.replace('India','Индия')

In [ ]:
data_df = data_df.pivot_table(values='Value'
                                       , index=['Country Name', 'Report Date']
                                       , columns='Indicator Name').reset_index(drop=False)

In [ ]:
data_df.rename(columns={'Country Name' : 'Страна'
                        , 'Report Date' : 'report_date'
                        , 'Market capitalization of listed domestic companies (constant 2015 US$)' : 'Капитализация'
                        , 'Stocks traded, total value (constant 2015 US$)' : 'Объем торгов'
                        , 'Domestic Credit (constant 2015 US$)' : 'Задолженность'
                        , 'Total Export Value (constant 2015 US$)' : 'Экспорт'
                        , 'GDP (constant 2015 US$)' : 'ВВП'
                       }
               , inplace=True)

In [ ]:
data_df = data_df[data_df['report_date'] >= '1999-12-31']
data_df = data_df[data_df['report_date'] <= '2025-03-31']

In [ ]:
data_df = data_df.set_index('report_date')

# Seasonal decomposition

In [ ]:
deseasoned_df = pd.DataFrame()

for country_name, country_df in data_df.groupby('Страна'):
    deseasoned_df_temp = pd.DataFrame()
    for column_name in country_df.columns[1:]:

        stl = STL(country_df[column_name], period = 4 ,seasonal=9, low_pass=21, robust=True)
        result = stl.fit()
    
        stl_df_temp = pd.DataFrame(result.observed - result.seasonal, columns=[column_name + ' (сезонность устранена) STL'])

        stl_df_temp = pd.merge(country_df[[column_name]], stl_df_temp, on = 'report_date', how='left')

        deseasoned_df_temp = pd.concat([deseasoned_df_temp, stl_df_temp], axis=1)

    deseasoned_df_temp['Страна'] = country_name

    deseasoned_df = pd.concat([deseasoned_df, deseasoned_df_temp])

In [ ]:
deseasoned_df = deseasoned_df.reset_index(drop=False
                                         ).melt(id_vars = ['Страна'
                                                           , 'report_date'
                                                          ]
                                                , value_vars = deseasoned_df.columns[:-1]
                                                , ignore_index=False)

In [ ]:
deseasoned_df['variable_country'] = deseasoned_df['Страна'] + ' - ' + deseasoned_df['variable']

In [ ]:
deseasoned_df.pivot_table(values='value'
                          , index=['report_date'
                                  ]
                          , columns=['variable'
                                     , 'Страна'
                                     , 'variable_country']).to_excel(r'explore_data\_time_series_decomposition_quarterly.xlsx', index=True)

In [ ]:
deseasoned_df = deseasoned_df[deseasoned_df['variable'].isin(['ВВП (сезонность устранена) STL'
                                                              , 'Капитализация (сезонность устранена) STL'
                                                              , 'Задолженность (сезонность устранена) STL'
                                                              , 'Экспорт (сезонность устранена) STL'
                                                             ]
                                                            )]

In [ ]:
deseasoned_df = deseasoned_df.pivot_table(values='value'
                                          , index=['report_date'
                                                   , 'Страна'
                                                  ]
                                          , columns=['variable']
                                         ).reset_index(drop=False)

In [ ]:
deseasoned_df.rename(columns={'ВВП (сезонность устранена) STL' : 'ВВП'
                              , 'Капитализация (сезонность устранена) STL' : 'Капитализация'
                              , 'Задолженность (сезонность устранена) STL' : 'Задолженность'
                              , 'Экспорт (сезонность устранена) STL' : 'Экспорт'
                             }
                     , inplace=True
                    )

In [ ]:
data_df = deseasoned_df.copy()

In [ ]:
data_df['index_val'] = data_df['Страна'].astype(str) + ' ' + data_df['report_date'].astype(str)

In [ ]:
data_df = data_df.sort_values(['Страна', 'report_date'])

In [ ]:
data_df = data_df[['report_date'
                   , 'Страна'
                   , 'ВВП'
                   , 'Капитализация'
                   , 'Задолженность'
                   , 'Экспорт'
                   , 'index_val'
                  ]]

# data_df = data_df[data_df.columns[6:-6]]

# Relationship

## Level

In [ ]:
level_data_df = data_df.copy()

In [ ]:
level_data_df = level_data_df[level_data_df['report_date'] >= '2000-03-31']

In [ ]:
# level_data_df = level_data_df[level_data_df['report_date'] >= '2000-03-31']
# level_data_df = level_data_df[level_data_df['report_date'] <= '2010-12-31']

In [ ]:
# level_data_df = level_data_df[level_data_df['report_date'] > '2010-12-31']

In [ ]:
level_data_df.melt(id_vars=['Страна', 'report_date'], value_vars=['ВВП'
                                                                  , 'Капитализация'
                                                                  , 'Задолженность'
                                                                  , 'Экспорт'
                                                                 ], ignore_index=False).to_excel(r'explore_data\_quarterly_const_USD_level.xlsx', index=False)

In [ ]:
level_data_df = level_data_df.set_index('report_date')

# level_data_df = level_data_df[['Страна'
#                                , 'ВВП'
#                                , 'Волатильность'
#                                , 'Цена/прибыль'
#                                , 'Капитализация'
#                                , 'Задолженность'
#                                , 'Экспорт'
#                               ]]

In [ ]:
num_of_obs = int(len(level_data_df.index)/2)

### china

In [ ]:
sns.set_theme(style="ticks", font_scale=1.35)

sns.pairplot(level_data_df[level_data_df['Страна']=='Китай'].reset_index(drop=False)

             , hue = 'report_date'
             , palette = sns.color_palette("Greys", num_of_obs)
             , markers =['$\circ$'] * num_of_obs
             
             , x_vars=['Капитализация'
                       , 'Задолженность'
                       , 'Экспорт'
                      ]
             , y_vars=['ВВП']
             
             , kind='reg'
             , height=3
             , plot_kws={'line_kws': {'linestyle': '--'
                                      , 'linewidth': 1
                                     }
                         , 'scatter_kws': {'s': 25
                                           , 'alpha': 0.7}
                        }
             , dropna=True
            )

plt.savefig('explore_data\_relationship_level_china.png')
plt.show()

### india

In [ ]:
sns.set_theme(style="ticks", font_scale=1.35)

sns.pairplot(level_data_df[level_data_df['Страна']=='Индия'].reset_index(drop=False)

             , hue = 'report_date'
             , palette = sns.color_palette("Greys", num_of_obs)
             , markers =['x'] * num_of_obs
             
             , x_vars=['Капитализация'
                       , 'Задолженность'
                       , 'Экспорт'
                      ]
             , y_vars=['ВВП']
             
             , kind='reg'
             , height=3
             , plot_kws={'line_kws': {'linestyle': '--'
                                      , 'linewidth': 1
                                     }
                         , 'scatter_kws': {'s': 25
                                           , 'alpha': 0.7}
                        }
             , dropna=True
            )

plt.savefig('explore_data\_relationship_level_india.png')
plt.show()

### both

In [ ]:
# sns.set_theme(style="ticks", font_scale=1.35)

# sns.pairplot(level_data_df.reset_index(drop=False)

#              , hue = 'Страна'
#              , palette = ['Black', 'Black']
#              , markers =['x', '$\circ$']
             
#              , x_vars=['Капитализация'
#                        , 'Задолженность'
#                        , 'Экспорт'
#                        , 'Волатильность'
#                        , 'Цена/прибыль'
#                       ]
#              , y_vars=['ВВП']
             
#              , kind='reg'
#              , height=3
#              , plot_kws={'line_kws': {'linestyle': '--'
#                                       , 'linewidth': 1
#                                      }
#                          , 'scatter_kws': {'s': 25
#                                            , 'alpha': 0.75}
#                         }
#              , dropna=True
#             )

In [ ]:
sns.set_theme(style="ticks", font_scale=1.35)

sns.pairplot(level_data_df.reset_index(drop=False)

             , hue = 'index_val'
             , palette = sns.color_palette("Greys", num_of_obs) + sns.color_palette("Greys", num_of_obs)
             , markers = ['x'] * num_of_obs + ['$\circ$'] * num_of_obs
             
             , x_vars=['Капитализация'
                       , 'Задолженность'
                       , 'Экспорт'
                      ]
             , y_vars=['ВВП']
             
             , kind='reg'
             , height=3
             , plot_kws={'line_kws': {'linestyle': '--'
                                      , 'linewidth': 1
                                     }
                         , 'scatter_kws': {'s': 25
                                           , 'alpha': 0.7}
                        }
             , dropna=True
            )

plt.savefig('explore_data\_relationship_level_both.png')
plt.show()

## Differenced

In [ ]:
diff_data_df = pd.DataFrame()

In [ ]:
for country_name, country_df in data_df.groupby('Страна'):
    # for column in country_df.columns[2:-1]:
    country_df[['ВВП'
                , 'Капитализация'
                , 'Задолженность'
                , 'Экспорт'
               ]] = country_df[['ВВП'
                                , 'Капитализация'
                                , 'Задолженность'
                                , 'Экспорт'
                               ]].diff()
    diff_data_df = pd.concat([diff_data_df, country_df])

In [ ]:
diff_data_df = diff_data_df[diff_data_df['report_date'] >= '2000-03-31']

In [ ]:
# diff_data_df = diff_data_df[diff_data_df['report_date'] >= '2000-03-31']
# diff_data_df = diff_data_df[diff_data_df['report_date'] <= '2010-12-31']

In [ ]:
# diff_data_df = diff_data_df[diff_data_df['report_date'] > '2010-12-31']

In [ ]:
diff_data_df.melt(id_vars=['Страна', 'report_date'], value_vars=['ВВП'
                                                                 , 'Капитализация'
                                                                 , 'Задолженность'
                                                                 , 'Экспорт'
                                                                ], ignore_index=False).to_excel(r'explore_data\_quarterly_const_USD_diff.xlsx', index=False)

In [ ]:
diff_data_df = diff_data_df.set_index('report_date')

### china

In [ ]:
sns.set_theme(style="ticks", font_scale=1.35)

sns.pairplot(diff_data_df[diff_data_df['Страна']=='Китай'].reset_index(drop=False)

             , hue = 'report_date'
             , palette = sns.color_palette("Greys", num_of_obs)
             , markers =['$\circ$'] * num_of_obs
             
             , x_vars=['Капитализация'
                       , 'Задолженность'
                       , 'Экспорт'
                      ]
             , y_vars=['ВВП']
             
             , kind='reg'
             , height=3
             , plot_kws={'line_kws': {'linestyle': '--'
                                      , 'linewidth': 1
                                     }
                         , 'scatter_kws': {'s': 25
                                           , 'alpha': 0.7}
                        }
             , dropna=True
            )

plt.savefig('explore_data\_relationship_diff_china.png')
plt.show()

### india

In [ ]:
sns.set_theme(style="ticks", font_scale=1.35)

sns.pairplot(diff_data_df[diff_data_df['Страна']=='Индия'].reset_index(drop=False)

             , hue = 'report_date'
             , palette = sns.color_palette("Greys", num_of_obs)
             , markers =['x'] * num_of_obs
             
             , x_vars=['Капитализация'
                       , 'Задолженность'
                       , 'Экспорт'
                      ]
             , y_vars=['ВВП']
             
             , kind='reg'
             , height=3
             , plot_kws={'line_kws': {'linestyle': '--'
                                      , 'linewidth': 1
                                     }
                         , 'scatter_kws': {'s': 25
                                           , 'alpha': 0.7}
                        }
             , dropna=True
            )

plt.savefig('explore_data\_relationship_diff_india.png')
plt.show()

### both

In [ ]:
# sns.set_theme(style="ticks", font_scale=1.35)

# sns.pairplot(diff_data_df.reset_index(drop=False)

#              , hue = 'Страна'
#              , palette = ['Black', 'Black']
#              , markers =['x', '$\circ$']
             
#              , x_vars=['Капитализация'
#                        , 'Задолженность'
#                        , 'Экспорт'
#                        , 'Волатильность'
#                        , 'Цена/прибыль'
#                       ]
#              , y_vars=['ВВП']
             
#              , kind='reg'
#              , height=3
#              , plot_kws={'line_kws': {'linestyle': '--'
#                                       , 'linewidth': 1
#                                      }
#                          , 'scatter_kws': {'s': 25
#                                            , 'alpha': 0.75}
#                         }
#              , dropna=True
#             )

In [ ]:
sns.set_theme(style="ticks", font_scale=1.35)

sns.pairplot(diff_data_df.reset_index(drop=False)

             , hue = 'index_val'
             , palette = sns.color_palette("Greys", num_of_obs) + sns.color_palette("Greys", num_of_obs)
             , markers = ['x'] * num_of_obs + ['$\circ$'] * num_of_obs
             
             , x_vars=['Капитализация'
                       , 'Задолженность'
                       , 'Экспорт'
                      ]
             , y_vars=['ВВП']
             
             , kind='reg'
             , height=3
             , plot_kws={'line_kws': {'linestyle': '--'
                                      , 'linewidth': 1
                                     }
                         , 'scatter_kws': {'s': 25
                                           , 'alpha': 0.7}
                        }
             , dropna=True
            )

plt.savefig('explore_data\_relationship_diff_both.png')
plt.show()

# Stationairty

## Level

In [ ]:
# level_data_df[level_data_df['Страна']=='Китай'][['ВВП'
#                                                  , 'Волатильность'
#                                                  , 'Цена/прибыль'
#                                                  , 'Капитализация'
#                                                  , 'Задолженность'
#                                                  , 'Экспорт'
#                                                 ]].plot(subplots=True, figsize=(10, 10))

In [ ]:
## buidling the df
stationarity_df = pd.DataFrame({
    'Country' : [],
    'Parameter' : [],
    'ADF p-value (level data)': [],
    'DFGLS p-value (level data)': [],
    'PhillipsPerron p-value (level data)': [],
    'KPSS p-value (level data)': [],
    # 'ZivotAndrews p-value (level data)': []
})


for country_name, country_df in level_data_df.groupby('Страна'):
    for column in country_df.columns[1:-1]:
    
        adf_result = ADF(country_df[column].dropna()
                         , lags = 1
                         , trend = 'c'
                         # , max_lags = 1
                         # , method = 'bic'
                         ).pvalue
    
        dfgls_result = DFGLS(country_df[column].dropna()
                             , lags = 1
                             , trend = 'c'
                             # , max_lags = 1
                             # , method = 'bic'
                            ).pvalue
    
        PhillipsPerron_result = PhillipsPerron(country_df[column].dropna()
                                               # , lags = 1
                                               , trend = 'c'
                                               , test_type = 'tau'
                                              ).pvalue
    
        kpss_result = kpss(country_df[column].dropna()
                           , regression='c'
                           , nlags='auto'
                           , store=False)[1]
    
        # ZivotAndrews_result = ZivotAndrews(country_df[column].dropna()
        #                                        # , lags = 1
        #                                        , trend = 'c'
        #                                        , trim = 0.15
        #                                        , max_lags = 1
        #                                        , method = 'bic'
        #                                       ).pvalue
        
        # buidling the df
        stationarity_temp_df = pd.DataFrame({
            'Country' : [country_name],
            'Parameter' : [column],# + f''' {str(min(country_df[[column]].dropna().index).strftime("%Y-%m-%d"))} - {str(max(country_df[[column]].dropna().index).strftime("%Y-%m-%d"))}'''],
            'ADF p-value (level data)': [round(adf_result, 4)],
            'DFGLS p-value (level data)': [round(dfgls_result, 4)],
            'PhillipsPerron p-value (level data)': [round(PhillipsPerron_result, 4)],
            'KPSS p-value (level data)': [round(kpss_result, 4)],
            # 'ZivotAndrews p-value (level data)': [round(ZivotAndrews_result, 4)]
        })
        
        stationarity_df = pd.concat([stationarity_df, stationarity_temp_df])

## Differenced

In [ ]:
# diff_data_df[diff_data_df['Страна']=='Китай'][['ВВП'
#                                                , 'Волатильность'
#                                                , 'Цена/прибыль'
#                                                , 'Капитализация'
#                                                , 'Задолженность'
#                                                , 'Экспорт'
#                                               ]].plot(subplots=True, figsize=(10, 10))

In [ ]:
# buidling the df
integrated_stationarity_df = pd.DataFrame({
    'Country' : [],
    'Parameter' : [],
    'ADF p-value (differenced data)': [],
    'DFGLS p-value (differenced data)': [],
    'PhillipsPerron p-value (differenced data)': [],
    'KPSS p-value (differenced data)': [],
    # 'ZivotAndrews p-value (differenced data)': []
})

for country_name, country_df in diff_data_df.groupby('Страна'):
    country_df = country_df.dropna()
    for column in country_df.columns[1:-1]:
    
        adf_result = ADF(country_df[column].dropna()
                         , lags = 1
                         , trend = 'c'
                         # , max_lags = 1
                         # , method = 'bic'
                         ).pvalue
    
        dfgls_result = DFGLS(country_df[column].dropna()
                             , lags = 1
                             , trend = 'c'
                             # , max_lags = 1
                             # , method = 'bic'
                            ).pvalue
    
        PhillipsPerron_result = PhillipsPerron(country_df[column].dropna()
                                               # , lags = 1
                                               , trend = 'c'
                                               , test_type = 'tau'
                                              ).pvalue
    
        kpss_result = kpss(country_df[column].dropna()
                           , regression='c'
                           , nlags='auto'
                           , store=False)[1]
    
        # ZivotAndrews_result = ZivotAndrews(country_df[column].dropna()
        #                                        # , lags = 1
        #                                        , trend = 'с'
        #                                        , trim = 0.15
        #                                        , max_lags = 1
        #                                        , method = 'bic'
        #                                       ).pvalue
        
        # buidling the df
        stationarity_temp_df = pd.DataFrame({
            'Country' : [country_name],
            'Parameter' : [column],# + f''' {str(min(country_df[[column]].dropna().index).strftime("%Y-%m-%d"))} - {str(max(country_df[[column]].dropna().index).strftime("%Y-%m-%d"))}'''],
            'ADF p-value (differenced data)': [round(adf_result, 4)],
            'DFGLS p-value (differenced data)': [round(dfgls_result, 4)],
            'PhillipsPerron p-value (differenced data)': [round(PhillipsPerron_result, 4)],
            'KPSS p-value (differenced data)': [round(kpss_result, 4)],
            # 'ZivotAndrews p-value (differenced data)': [round(ZivotAndrews_result, 4)]
        })
        
        integrated_stationarity_df = pd.concat([integrated_stationarity_df, stationarity_temp_df])

## Together & export

Augmented Dickey-Fuller
The null hypothesis of the Augmented Dickey-Fuller is that there is a unit root, with the alternative that there is no unit root. 

Dickey-Fuller GLS
The null hypothesis of the Dickey-Fuller GLS is that there is a unit root, with the alternative that there is no unit root.

Phillips-Perron
The null hypothesis of the Phillips-Perron (PP) test is that there is a unit root, with the alternative that there is no unit root. 

KPSS
The null hypothesis of the KPSS test is that the series is weakly stationary and the alternative is that it is non-stationary. 

In [ ]:
pd.merge(stationarity_df, integrated_stationarity_df, on =['Country', 'Parameter'], how='left')

In [ ]:
pd.merge(stationarity_df, integrated_stationarity_df, on =['Country', 'Parameter'], how='left').to_excel(r'explore_data\_stationarity_analysis.xlsx', index=False)

# Cointegration (level)

In [ ]:
coint_data_df = level_data_df.reset_index()[['report_date'
                                             , 'Страна'
                                             , 'ВВП'
                                             , 'Капитализация'
                                             , 'Задолженность'
                                             , 'Экспорт'
                                            ]]

In [ ]:
coint_test_df = pd.DataFrame()

for country_name, country_df in coint_data_df.groupby('Страна'):
    # print(country_name)
    country_df = country_df.set_index('report_date')
    
    for pairs in combinations(list(country_df.columns[1:]), 2):
        # print(pairs)
        temp_df = country_df[[pairs[0], pairs[1]]].dropna()
    
    
        result = coint(temp_df[pairs[0]], temp_df[pairs[1]]
                  , trend='ct'
                      )
        
        # creating a unit name to unit value dictionary
        coint_test = {'var_1': [pairs[0]]
                      , 'var_2': [pairs[1]]
                      , 'p_value': [result[1]]
                     }
        temp_coint_test_df = pd.DataFrame.from_dict(coint_test)
        
        temp_coint_test_df['country'] = country_name
        
        coint_test_df = pd.concat([coint_test_df, temp_coint_test_df])

In [ ]:
# making a matrix of results
coint_test_df = coint_test_df.pivot_table(values='p_value'
                                     , index=['var_2']
                                     , columns=['var_1', 'country'])

# Sort the DataFrame by the number of non-null values
coint_test_df['non_null_count'] = coint_test_df.count(axis=1) # Calculate non-null counts per row
coint_test_df = coint_test_df.sort_values(by='non_null_count', ascending=False) # Sort in descending order
coint_test_df = coint_test_df.drop('non_null_count', axis=1)

# Sort the DataFrame by the number of non-null values
coint_test_df = coint_test_df.T
coint_test_df['non_null_count'] = coint_test_df.count(axis=1) # Calculate non-null counts per row
coint_test_df = coint_test_df.sort_values(by='non_null_count', ascending=False) # Sort in descending order
coint_test_df = coint_test_df.drop('non_null_count', axis=1)

In [ ]:
coint_test_df = coint_test_df.T

In [ ]:
coint_test_df = coint_test_df[::-1]

In [ ]:
coint_test_df

In [ ]:
coint_test_df.to_excel(r'explore_data\_coint_test.xlsx', index=True)

# VAR (differenced)

## India

In [ ]:
country_name = 'Индия'
diff_country_df = diff_data_df[diff_data_df['Страна']==country_name]

diff_country_df = diff_country_df[['ВВП'
                                   , 'Капитализация'
                                   , 'Задолженность'
                                   , 'Экспорт'
                                  ]]

# Create the VAR model
model = VAR(endog = diff_country_df
           )  # or VAR(data) if your series were already stationary

# Select optimal lag order
lag_number = model.select_order(trend='c').selected_orders['aic']

# FULL data VAR maxlags=4, trend='n'

# Fit the model with the selected lag order
var_model = model.fit(maxlags=lag_number
                      , ic=None
                      , trend='c'
                     )

# Summary of the model
print(var_model.summary())

In [ ]:
macro_variable = 'ВВП'

plt.figure(figsize=(6,3))

plt.plot(diff_country_df[macro_variable])
plt.plot(var_model.fittedvalues[macro_variable])


plt.legend(('Факт', 'Модель'), fontsize=16)
plt.title(f'''Факт и прогноз {macro_variable}''', fontsize=14, pad=15)

plt.savefig('explore_data\_gdp_obs_model_india.png')
plt.show()

### Forecast

In [ ]:
forecast_steps_val = 16

point, lower, upper = var_model.forecast_interval(y = diff_country_df.values
                                                  , steps = forecast_steps_val
                                                  , alpha=0.05
                                                 )

point_df = pd.DataFrame(point, columns = diff_country_df.columns
                       )

lower_df = pd.DataFrame(lower, columns = diff_country_df.columns
                       )

upper_df = pd.DataFrame(upper, columns = diff_country_df.columns
                       )


point_df.index = pd.date_range(start=max(diff_country_df.index), periods=forecast_steps_val + 1, freq='Q')[1:]
lower_df.index = pd.date_range(start=max(diff_country_df.index), periods=forecast_steps_val + 1, freq='Q')[1:]
upper_df.index = pd.date_range(start=max(diff_country_df.index), periods=forecast_steps_val + 1, freq='Q')[1:]

In [ ]:
fig, ax = plt.subplots(figsize=(6,3))

# Plot with labels for legend
ax.plot(pd.concat([diff_country_df
                   , point_df
                  ]
                 )[macro_variable]
        , label=macro_variable
       )

ax.plot(pd.DataFrame(np.zeros(len(pd.concat([diff_country_df
                                             , point_df
                                            ]
                                           )[macro_variable].index
                                 )
                             )
                     , index=pd.concat([diff_country_df
                                        , point_df
                                       ]
                                      )[macro_variable].index
                    )
        , label='Нулевая линия'
        , linestyle='--'
       )

ax.fill_between(point_df[macro_variable].index
                , lower_df[macro_variable]
                , upper_df[macro_variable]
                , color='b'
                , alpha=.2
                , label='Доверительный интервал'
               )

# Add title and axis labels
ax.set_title(f'''Прогноз приростов {macro_variable}''', fontsize=14, pad=15)
ax.set_xlabel('Дата/Время', fontsize=12)
ax.set_ylabel(macro_variable, fontsize=12)

# Add legend
ax.legend(loc='best', fontsize=10)

# # Optional: Customize grid
# ax.grid(True, alpha=0.3)

# Optional: Adjust layout to prevent clipping
plt.tight_layout()

plt.savefig('explore_data\_gdp_forecast_diff_india.png')
plt.show()

In [ ]:
level_country_df = level_data_df[level_data_df['Страна'] == country_name]

level_country_df = level_country_df[diff_country_df.columns]

fig, ax = plt.subplots(figsize=(6, 3))

# Plot with labels for legend
ax.plot(level_country_df[macro_variable]
        , label=macro_variable
       )

ax.plot(pd.concat([level_country_df.iloc[-1:]
                   , point_df
                  ]
                 ).cumsum()[macro_variable]
        , label=macro_variable + ' (прогноз)'
       )

ax.fill_between(pd.concat([level_country_df.iloc[-1:]
                           , point_df
                          ]
                         ).cumsum()[macro_variable].index
                
                , pd.concat([level_country_df.iloc[-1:]
                             , lower_df
                            ]
                           ).cumsum()[macro_variable]
                
                , pd.concat([level_country_df.iloc[-1:]
                             , upper_df
                            ]
                           ).cumsum()[macro_variable]
                
                , color='brown'
                , alpha=.2
                , label='Доверительный интервал'
               )


# Add title and axis labels
ax.set_title(f'''Прогноз значения {macro_variable}''', fontsize=14, pad=15)
ax.set_xlabel('Дата/Время', fontsize=12)
ax.set_ylabel(macro_variable, fontsize=12)

# Add legend
ax.legend(loc='best', fontsize=10)

# # Optional: Customize grid
# ax.grid(True, alpha=0.3)

# Optional: Adjust layout to prevent clipping
plt.tight_layout()

plt.savefig('explore_data\_gdp_forecast_value_india.png')
plt.show()

### IRF

In [ ]:
# Impulse response analysis
irf = var_model.irf(periods=36)
irf.plot(figsize=(25, 25)
         # , response='ВВП'
         , orth=False
        )
# plt.tight_layout()
plt.savefig('explore_data\_irf_india.png')
plt.show()

### CIRF

In [ ]:
irf.plot_cum_effects(figsize=(25, 25)
                    # , response='ВВП'
         # , plot_stderr = False
                    )
plt.savefig('explore_data\_cirf_india.png')
plt.show()

### FEVD

In [ ]:
# Variance decomposition
fevd = var_model.fevd(periods=36)
# print(fevd.summary())

# Plot FEVD
fevd.plot(figsize=(18, 21))
# plt.tight_layout()
plt.savefig('explore_data\_fevd_india.png')
plt.show()

### Granger

In [ ]:
lagvar = 8

In [ ]:
ssr_based_chi2_df = pd.DataFrame()

for macro_variable in diff_country_df.columns[1:]:
    print("="*10 + macro_variable + "="*10)

    test = grangercausalitytests(diff_country_df[[diff_country_df.columns[0]
                                               , macro_variable]]
                                 , lagvar)

    temp_ssr_based_chi2_df = pd.DataFrame([test[1][0]['ssr_chi2test'][1]] + [test[2][0]['ssr_chi2test'][1]] + [test[3][0]['ssr_chi2test'][1]] + [test[4][0]['ssr_chi2test'][1]] + [test[5][0]['ssr_chi2test'][1]]
                                      + [test[6][0]['ssr_chi2test'][1]] + [test[7][0]['ssr_chi2test'][1]] + [test[8][0]['ssr_chi2test'][1]]
                                      , columns=[macro_variable]).T

    ssr_based_chi2_df = pd.concat([ssr_based_chi2_df, temp_ssr_based_chi2_df]) 
    print("\n"*5)

In [ ]:
ssr_based_chi2_df = ssr_based_chi2_df.T
ssr_based_chi2_df.index = ssr_based_chi2_df.index+1
ssr_based_chi2_df = ssr_based_chi2_df.reset_index(names='Кол-во лагов', drop=False)

In [ ]:
ssr_based_chi2_df.T.to_excel(r'explore_data\_granger_causality_india.xlsx', index=True)

In [ ]:
ssr_based_chi2_df

## China

In [ ]:
country_name = 'Китай'
diff_country_df = diff_data_df[diff_data_df['Страна']==country_name]

diff_country_df = diff_country_df[['ВВП'
                                   , 'Капитализация'
                                   , 'Задолженность'
                                   , 'Экспорт'
                                   ]]

# Create the VAR model
model = VAR(endog = diff_country_df
           )  # or VAR(data) if your series were already stationary

# Select optimal lag order
lag_number = model.select_order(trend='c').selected_orders['aic']

# FULL data VAR maxlags=4, trend='n'

# Fit the model with the selected lag order
var_model = model.fit(maxlags=lag_number
                      , ic=None
                      , trend='c'
                     )

# Summary of the model
print(var_model.summary())

In [ ]:
macro_variable = 'ВВП'

plt.figure(figsize=(6,3))

plt.plot(diff_country_df[macro_variable])
plt.plot(var_model.fittedvalues[macro_variable])


plt.legend(('Факт', 'Модель'), fontsize=16)
plt.title(f'''Факт и прогноз {macro_variable}''', fontsize=14, pad=15)

plt.savefig('explore_data\_gdp_obs_model_china.png')
plt.show()

### Forecast

In [ ]:
forecast_steps_val = 16

point, lower, upper = var_model.forecast_interval(y = diff_country_df.values
                                                  , steps = forecast_steps_val
                                                  , alpha=0.05
                                                 )

point_df = pd.DataFrame(point, columns = diff_country_df.columns
                       )

lower_df = pd.DataFrame(lower, columns = diff_country_df.columns
                       )

upper_df = pd.DataFrame(upper, columns = diff_country_df.columns
                       )


point_df.index = pd.date_range(start=max(diff_country_df.index), periods=forecast_steps_val + 1, freq='Q')[1:]
lower_df.index = pd.date_range(start=max(diff_country_df.index), periods=forecast_steps_val + 1, freq='Q')[1:]
upper_df.index = pd.date_range(start=max(diff_country_df.index), periods=forecast_steps_val + 1, freq='Q')[1:]

In [ ]:
fig, ax = plt.subplots(figsize=(6,3))

# Plot with labels for legend
ax.plot(pd.concat([diff_country_df
                   , point_df
                  ]
                 )[macro_variable]
        , label=macro_variable
       )

ax.plot(pd.DataFrame(np.zeros(len(pd.concat([diff_country_df
                                             , point_df
                                            ]
                                           )[macro_variable].index
                                 )
                             )
                     , index=pd.concat([diff_country_df
                                        , point_df
                                       ]
                                      )[macro_variable].index
                    )
        , label='Нулевая линия'
        , linestyle='--'
       )

ax.fill_between(point_df[macro_variable].index
                , lower_df[macro_variable]
                , upper_df[macro_variable]
                , color='b'
                , alpha=.2
                , label='Доверительный интервал'
               )

# Add title and axis labels
ax.set_title(f'''Прогноз приростов {macro_variable}''', fontsize=14, pad=15)
ax.set_xlabel('Дата/Время', fontsize=12)
ax.set_ylabel(macro_variable, fontsize=12)

# Add legend
ax.legend(loc='best', fontsize=10)

# # Optional: Customize grid
# ax.grid(True, alpha=0.3)

# Optional: Adjust layout to prevent clipping
plt.tight_layout()

plt.savefig('explore_data\_gdp_forecast_diff_china.png')
plt.show()

In [ ]:
level_country_df = level_data_df[level_data_df['Страна'] == country_name]

level_country_df = level_country_df[diff_country_df.columns]

fig, ax = plt.subplots(figsize=(6, 3))

# Plot with labels for legend
ax.plot(level_country_df[macro_variable]
        , label=macro_variable
       )

ax.plot(pd.concat([level_country_df.iloc[-1:]
                   , point_df
                  ]
                 ).cumsum()[macro_variable]
        , label=macro_variable + ' (прогноз)'
       )

ax.fill_between(pd.concat([level_country_df.iloc[-1:]
                           , point_df
                          ]
                         ).cumsum()[macro_variable].index
                
                , pd.concat([level_country_df.iloc[-1:]
                             , lower_df
                            ]
                           ).cumsum()[macro_variable]
                
                , pd.concat([level_country_df.iloc[-1:]
                             , upper_df
                            ]
                           ).cumsum()[macro_variable]
                
                , color='brown'
                , alpha=.2
                , label='Доверительный интервал'
               )


# Add title and axis labels
ax.set_title(f'''Прогноз значения {macro_variable}''', fontsize=14, pad=15)
ax.set_xlabel('Дата/Время', fontsize=12)
ax.set_ylabel(macro_variable, fontsize=12)

# Add legend
ax.legend(loc='best', fontsize=10)

# # Optional: Customize grid
# ax.grid(True, alpha=0.3)

# Optional: Adjust layout to prevent clipping
plt.tight_layout()

plt.savefig('explore_data\_gdp_forecast_value_china.png')
plt.show()

### IRF

In [ ]:
# Impulse response analysis
irf = var_model.irf(periods=36)
irf.plot(figsize=(25, 25)
         # , response='ВВП'
         , orth=False
        )
# plt.tight_layout()
plt.savefig('explore_data\_irf_china.png')
plt.show()

### CIRF

In [ ]:
irf.plot_cum_effects(figsize=(25, 25)
                    # , response='ВВП'
         # , plot_stderr = False
                    )
plt.savefig('explore_data\_cirf_china.png')
plt.show()

### FEVD

In [ ]:
# Variance decomposition
fevd = var_model.fevd(periods=36)
# print(fevd.summary())

# Plot FEVD
fevd.plot(figsize=(18, 21))
# plt.tight_layout()
plt.savefig('explore_data\_fevd_china.png')
plt.show()

### Granger

In [ ]:
lagvar = 8

In [ ]:
ssr_based_chi2_df = pd.DataFrame()

for macro_variable in diff_country_df.columns[1:]:
    print("="*10 + macro_variable + "="*10)

    test = grangercausalitytests(diff_country_df[[diff_country_df.columns[0]
                                               , macro_variable]]
                                 , lagvar)

    temp_ssr_based_chi2_df = pd.DataFrame([test[1][0]['ssr_chi2test'][1]] + [test[2][0]['ssr_chi2test'][1]] + [test[3][0]['ssr_chi2test'][1]] + [test[4][0]['ssr_chi2test'][1]] + [test[5][0]['ssr_chi2test'][1]]
                                      + [test[6][0]['ssr_chi2test'][1]] + [test[7][0]['ssr_chi2test'][1]] + [test[8][0]['ssr_chi2test'][1]]
                                      , columns=[macro_variable]).T

    ssr_based_chi2_df = pd.concat([ssr_based_chi2_df, temp_ssr_based_chi2_df]) 
    print("\n"*5)

In [ ]:
ssr_based_chi2_df = ssr_based_chi2_df.T
ssr_based_chi2_df.index = ssr_based_chi2_df.index+1
ssr_based_chi2_df = ssr_based_chi2_df.reset_index(names='Кол-во лагов', drop=False)

In [ ]:
ssr_based_chi2_df.T.to_excel(r'explore_data\_granger_causality_china.xlsx', index=True)

In [ ]:
ssr_based_chi2_df